# 🚨 Real-Time Motorcycle Accident Detection & Telegram Notification System
### With Adaptive Low-Light Enhancement (CIE LAB + Gamma + CLAHE) and ByteTrack

โน้ตบุ๊กนี้เชื่อมต่อกับ GitHub Repository [`tormisster/accident-detection`](https://github.com/tormisster/accident-detection) เพื่อรันระบบตรวจจับอุบัติเหตุรถจักรยานยนต์และแจ้งเตือนเข้า Telegram บน **Google Colab (GPU)** ได้ทันที

## 1. ดึงโปรเจกต์จาก GitHub และติดตั้งไลบรารี (Clone & Install Dependencies)

In [ ]:
import os

REPO_URL = "https://github.com/tormisster/accident-detection.git"
REPO_DIR = "/content/accident-detection"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
!pip install -q -r requirements.txt
print("✅ ติดตั้งไลบรารีและเตรียมโปรเจกต์เรียบร้อยแล้ว!")

## 2. ตั้งค่า Telegram Bot (`@Accident_accident_bot`) และพารามิเตอร์โมเดล
เชื่อมต่อกับบอท `@Accident_accident_bot` (`Chat ID: 6819938549`) ไว้ให้พร้อมใช้งานทันทีเมื่อกดรัน

In [ ]:
import yaml

#@title ตั้งค่า Telegram & พารามิเตอร์การตรวจจับ
TELEGRAM_BOT_TOKEN = "9233730263:AAEkjMzC-pK2PO7ekTGgN4rmVYtqK3hHqHo" #@param {type:"string"}
TELEGRAM_CHAT_ID = "6819938549,9191991912,-5505596792" #@param {type:"string"}
CONF_THRESHOLD = 0.50 #@param {type:"slider", min:0.20, max:0.90, step:0.05}
LOW_LIGHT_THRESHOLD = 90.0 #@param {type:"slider", min:40.0, max:150.0, step:5.0}
CONFIRMATION_FRAMES = 5 #@param {type:"integer"}

with open("config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["model"]["conf_threshold"] = float(CONF_THRESHOLD)
cfg["preprocessing"]["low_light_threshold"] = float(LOW_LIGHT_THRESHOLD)
cfg["temporal_confirmation"]["confirmation_frames"] = int(CONFIRMATION_FRAMES)
cfg["telegram"]["enabled"] = True
cfg["telegram"]["bot_token"] = TELEGRAM_BOT_TOKEN.strip()
cfg["telegram"]["chat_id"] = str(TELEGRAM_CHAT_ID).strip()

with open("config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("✅ เชื่อมต่อ @Accident_accident_bot และอัปเดต config.yaml เรียบร้อยแล้ว!")

## 3. รันระบบตรวจจับอุบัติเหตุกับคลิปกล้องวงจรปิด (Run Inference on CCTV Clip)
- คลิปทดสอบเริ่มต้น: `ภาพคลิปอุบัติเหตุ จากกล้องวงจรปิด สี่แยก บ่อขิง ม.7 ต.หน้าพระธาตุ อ.พนัสนิคม ชลบุรี.mp4`
- ตั้งค่า `START_FRAME = 1145` เพื่อเริ่มประมวลผลช่วงเกิดเหตุชนทันที (หรือตั้งเป็น `0` เพื่อรันตั้งแต่ต้นคลิป)

In [ ]:
#@title รันไปป์ไลน์ตรวจจับอุบัติเหตุ (Low-Light + YOLO11x + ByteTrack + Telegram)
VIDEO_SOURCE = "ภาพคลิปอุบัติเหตุ จากกล้องวงจรปิด สี่แยก บ่อขิง ม.7 ต.หน้าพระธาตุ อ.พนัสนิคม ชลบุรี.mp4" #@param {type:"string"}
START_FRAME = 1145 #@param {type:"integer"}
MAX_FRAMES = 150 #@param {type:"integer"}
OUTPUT_VIDEO = "logs/colab_output.mp4"

!python3 inference.py \
    --config config.yaml \
    --source "{VIDEO_SOURCE}" \
    --start-frame {START_FRAME} \
    --max-frames {MAX_FRAMES} \
    --output "{OUTPUT_VIDEO}" \
    --no-display

## 4. แสดงภาพ Snapshot จังหวะยืนยันอุบัติเหตุ และเล่นคลิปวิดีโอผลลัพธ์บน Colab

In [ ]:
import glob
from IPython.display import Image, display, HTML
from base64 import b64encode

snapshots = sorted(glob.glob("logs/snapshots/accident_frame_*.jpg"))
if snapshots:
    latest_snap = snapshots[-1]
    print(f"📸 ภาพหลักฐานอุบัติเหตุล่าสุด: {latest_snap}")
    display(Image(filename=latest_snap, width=850))
else:
    print("ไม่พบภาพ Snapshot อุบัติเหตุในช่วงเฟรมที่เลือก")

In [ ]:
# แปลงวิดีโอผลลัพธ์เป็น H.264 เพื่อเล่นบนหน้าเว็บ Google Colab ได้โดยตรง
import os
from IPython.display import HTML
from base64 import b64encode

if os.path.exists("logs/colab_output.mp4"):
    !ffmpeg -y -i logs/colab_output.mp4 -vcodec libx264 -pix_fmt yuv420p logs/colab_output_h264.mp4 -loglevel quiet
    mp4 = open("logs/colab_output_h264.mp4", "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=850 controls autoplay loop>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))

## 5. ตารางบันทึกเหตุการณ์อุบัติเหตุและรัน Ablation Study สำหรับงานวิจัย

In [ ]:
import pandas as pd
from IPython.display import display

print("=== 1. ตารางบันทึกเหตุการณ์อุบัติเหตุ (logs/accident_events.csv) ===")
if os.path.exists("logs/accident_events.csv"):
    display(pd.read_csv("logs/accident_events.csv").tail(5))

print("\n=== 2. รันการทดลอง Ablation Study เปรียบเทียบ 4 รูปแบบ ===")
!python3 evaluate_ablation.py --config config.yaml --output-csv logs/ablation_results.csv --synthetic
if os.path.exists("logs/ablation_results.csv"):
    display(pd.read_csv("logs/ablation_results.csv"))